## System, User, and Assistant Roles

LLM conversations are not one continuous blob of text — they are structured as a sequence of labelled messages, each tagged with a role. The role tells the model who is "speaking" and changes how much authority that message carries.

## How it actually works

**The three core roles:**

**System** — instructions about how the model should behave overall: its persona,
constraints, tone, what it should and should not do. This is set once, typically by
the developer building the application, not by the end user. It carries the HIGHEST
authority — it is meant to shape everything else that follows.

**User** — what the actual human (or the application acting on their behalf) is
asking or saying. This is the role that changes with every new question or request.

**Assistant** — the model's own previous responses. When you have a multi-turn
conversation, every prior response the model gave gets included again as an
"assistant" message, so the model has the full conversation history to work from.

**Why this structure matters — it is not just formatting, it changes behaviour:**
A well-trained model is built to weigh these roles differently. Instructions in the
system role are generally treated as more authoritative and persistent than instructions
embedded in a user message. This is precisely WHY the system prompt is the right place
to put core behavioural rules (like your Personal Brand Autopilot's DRAFTER_PROMPT,
VOICE_GUARDIAN_PROMPT, etc.) rather than repeating those instructions inside every
single user message.

**Multi-turn conversations are built by replaying the whole role sequence each time:**
LLMs are stateless between API calls (covered in your LLM Systems quiz). Every single
request you send needs to include the FULL sequence of roles and messages so far —
system prompt, then user, assistant, user, assistant, and so on — because the model
has no memory of previous calls on its own. Your application is responsible for
reconstructing and resending that whole sequence every time.

**Why this directly explains your own code structure:**
In your Personal Brand Autopilot, every agent call sends `system=SOME_PROMPT` separately
from the `messages=[{"role": "user", "content": ...}]` list — that separation is not
arbitrary, it is exactly this role structure. The system prompt persists as the
authoritative instruction set; the user message is the specific task for that call.

## Where it breaks or gets dangerous

**Confusing system-level authority with user-level authority is a real security gap:**
This connects directly to prompt injection (covered in ml-fundamentals notebook 13).
If your application ever takes untrusted external content — a retrieved document,
a user-uploaded file, scraped web content — and inserts it into the conversation
in a way that LOOKS like it has system-level authority, or even just blends
confusingly with legitimate instructions, the model can be misled into treating
malicious instructions as if they came from a trusted source.

**The honest nuance: system role is influential, not absolutely unbreakable:**
A well-designed system prompt significantly increases the LIKELIHOOD a model follows
those instructions over conflicting user instructions, but it is not an unbreakable lock.
This is exactly why guardrails (covered later in this folder) exist as an independent
enforcement layer — you cannot rely purely on "I put it in the system prompt" as a
complete security guarantee for genuinely high-stakes constraints.

**Forgetting to resend the full role history breaks multi-turn coherence:**
If your application accidentally only sends the latest user message without the prior
assistant responses and earlier user messages, the model has genuinely lost all memory
of the conversation — not because it "forgot," but because that information was simply
never included in this specific request. This is a common bug source in any chat
application, and directly explains why your RAG Assistant and Finance Tracker need
explicit conversation history management (connects to context window management,
ml-fundamentals notebook 14).

**Direct, honest check on your own projects:**
Your Personal Brand Autopilot's agents are each STATELESS single-turn calls by design —
Agent 1 doesn't need Agent 2's conversation history, it needs Agent 1's OUTPUT passed
as fresh input to Agent 2. This is architecturally correct for a pipeline, but worth
being precise about in an interview: it is a deliberate choice that this is NOT a
multi-turn conversation needing role-history management, distinct from your RAG Assistant
or Finance Tracker, which genuinely ARE multi-turn and do need that management.

In [ ]:
# System, User, Assistant Roles — illustrating how the structure actually works

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("=== Single-turn call: system + one user message ===")
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=50,
    system="You are a terse D365 consultant. Answer in one short sentence, no fluff.",
    messages=[
        {"role": "user", "content": "What is three-way matching?"}
    ]
)
print("Assistant:", response.content[0].text)
print()

print("=== Multi-turn call: manually rebuilding the FULL role history ===")
print("(This is what your application must do every single time — the API has no memory)")
print()

conversation_history = [
    {"role": "user", "content": "What is three-way matching?"},
    {"role": "assistant", "content": "Three-way matching compares PO, receipt, and invoice quantities and prices."},
    {"role": "user", "content": "What happens if it fails?"}
]

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=60,
    system="You are a terse D365 consultant. Answer in one short sentence, no fluff.",
    messages=conversation_history
)
print("Full history sent (3 messages):")
for msg in conversation_history:
    print(f"  [{msg['role']}]: {msg['content'][:60]}")
print()
print("New assistant response:", response.content[0].text)
print()
print("Notice: the model correctly answered 'what happens if IT fails' —")
print("understanding 'it' refers to three-way matching — only because the full")
print("prior exchange was included in THIS request. Drop that history, and the")
print("model would have no idea what 'it' refers to at all.")

## My D365 analogy

The system/user/assistant role structure is similar to the difference between a
standing policy document, a specific transaction being processed, and the system's
own prior responses sitting in an audit trail.

The system role is like a standing approval policy configured for the legal entity —
it persists, it shapes how everything is handled, and it was set up deliberately by
someone with the authority to define how the process should work. It is not something
a random transaction gets to override just by showing up and asking nicely.

The user role is like the specific invoice being submitted right now — the actual
thing being acted on in this particular instance, distinct from the standing policy
that governs how it gets handled.

The assistant role, in a multi-turn exchange, is like the system's own prior decisions
sitting in the audit trail — if a vendor submits a follow-up invoice referencing
"the same issue as before," you need that audit trail of prior decisions available
again, in full, for the new decision to make any sense at all. Without resending that
full history, the new request is processed as if it has no relationship to anything
that came before it.

## LinkedIn post idea

A language model conversation is not one continuous block of text. It is a structured
sequence of labelled messages — system, user, assistant — and the label genuinely
changes how much weight each message carries.

The system message sets the standing rules. The user message is the specific request
right now. The assistant message is the model's own prior responses, replayed back
into every new request, because the model itself has no memory between API calls at all.

That last part surprised me when I actually understood it properly: every single
multi-turn conversation you have ever had with an AI assistant involved your entire
conversation history being resent, in full, with every new message. There is no
persistent memory living inside the model between calls. The illusion of an ongoing
conversation is built entirely by the application replaying the full history each time.

Once you see it this way, a lot of "why did the AI forget what we were talking about"
bugs stop being mysterious. Somewhere, the full history simply was not sent.

It is not that the model forgot. It is that it was never told.